In [1]:
!pip install -q numpy scikit-learn sentence-transformers


In [2]:
%%writefile lab6_search.py
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
class Retriever:
    def __init__(self, records, backend="tfidf"):
        if not records or any(not r["text"].strip() for r in records):
            raise ValueError("Provide nonempty records")
        self.records, self.backend = records, backend
        texts = [r["text"] for r in records]
        if backend == "tfidf":
            self.vectorizer = TfidfVectorizer()
            self.matrix = self.vectorizer.fit_transform(texts)
        elif backend == "embedding":
            from sentence_transformers import SentenceTransformer
            self.model = SentenceTransformer(
                "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
            self.check_length(texts)
            self.matrix = self.model.encode(
                texts, normalize_embeddings=True)
        else:
            raise ValueError("Use tfidf or embedding")
    def check_length(self, texts):
        for text in texts:
            n = len(self.model.tokenizer(text, truncation=False)
                    ["input_ids"])
            if n > self.model.max_seq_length:
                raise ValueError("Input exceeds encoder token limit")
    def search(self, query, k=3, filters=None):
        if not query.strip() or k <= 0:
            return []
        if self.backend == "tfidf":
            q = self.vectorizer.transform([query])
            if q.nnz == 0:
                return []
            scores = (q @ self.matrix.T).toarray()[0]
        else:
            self.check_length([query])
            q = self.model.encode([query], normalize_embeddings=True)[0]
            scores = self.matrix @ q
        allowed = [i for i, r in enumerate(self.records)
                   if all(r.get(key) == value
                          for key, value in (filters or {}).items())]
        ranked = sorted(allowed, key=lambda i: (-float(scores[i]), i))
        return [(self.records[i], float(scores[i])) for i in ranked[:k]]
def show_hits(hits):
    if not hits:
        print("No matching records.")
    for record, score in hits:
        print(record["id"], f"score={score:.3f}")
        print(record["text"])

Writing lab6_search.py


In [3]:
%%writefile task2_chunks.py
import re
def word_spans(text):
    return [(m.start(), m.end()) for m in re.finditer(r"\S+", text)]
def windows(text, size=40, overlap=10, offset=0):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size")
    spans = word_spans(text)
    result = []
    for i in range(0, len(spans), size - overlap):
        j = min(i + size, len(spans))
        result.append((offset + spans[i][0], offset + spans[j - 1][1]))
        if j == len(spans):
            break
    return result
def sentence_chunks(text, size=40):
    if size <= 0:
        raise ValueError("size must be positive")
    # Teaching splitter: punctuation followed by whitespace.
    cuts = [0] + [m.end() for m in
                  re.finditer(r"[.!?]\s+", text)] + [len(text)]
    out, pending, count = [], None, 0
    for a, b in zip(cuts, cuts[1:]):
        segment = text[a:b]
        n = len(segment.split())
        if not n:
            continue
        if n > size:
            if pending is not None:
                out.append(pending)
            out.extend(windows(segment, size, 0, a))
            pending, count = None, 0
        elif pending is not None and count + n <= size:
            pending, count = (pending[0], b), count + n
        else:
            if pending is not None:
                out.append(pending)
            pending, count = (a, b), n
    if pending is not None:
        out.append(pending)
    return out
def make_chunks(docs, strategy):
    result = []
    for doc_id, text in docs.items():
        spans = (sentence_chunks(text) if strategy == "sentence"
                 else windows(text, 40, int(strategy)))
        for i, (a, b) in enumerate(spans):
            result.append(dict(id=f"{doc_id}:{i}", doc=doc_id,
                               start=a, end=b, text=text[a:b]))
    return result
if __name__ == "__main__":
    assert windows("") == []
    assert windows("short text") == [(0, 10)]
    try:
        windows("a b", 2, 2)
    except ValueError:
        pass
    else:
        raise AssertionError("Invalid overlap was accepted")
    long = " ".join(["word"] * 95) + "."
    assert all(len(long[a:b].split()) <= 40
               for a, b in sentence_chunks(long))
    docs = {"example": "Reset your password. Keep recovery codes private."}
    for strategy in ["0", "10", "sentence"]:
        for c in make_chunks(docs, strategy):
            assert c["text"] == docs[c["doc"]][c["start"]:c["end"]]
            print(strategy, c)

Writing task2_chunks.py


In [4]:
%%writefile task2_compare.py
from task2_chunks import make_chunks, windows, sentence_chunks

# Three synthetic documents (150-250 words each). Synthetic teaching data.
DOCS = {
    "late": (
        "Assessment Policy. Assignments are due at 11:59 PM on the date "
        "stated in the course outline. Late submission is normally "
        "penalized by ten percent for each day after the deadline. "
        "Illness extensions require medical evidence before the deadline. "
        "Students must send the certificate to the course coordinator and "
        "wait for a written reply before assuming that an extension has "
        "been granted. Extensions are not given for travel plans or for a "
        "heavy workload in other courses. Group projects follow the same "
        "rule, and each member must report any illness separately to the "
        "coordinator. When a student submits work after the final accepted "
        "date, the coordinator records the submission time, compares it "
        "with the approved extension if there is one, applies the penalty "
        "for every full day of delay, and sends the final mark to the "
        "student together with a short note explaining how the penalty "
        "was calculated. Students may ask for a review of the penalty "
        "within five working days. Marks are published on the learning "
        "portal after all submissions have been checked."),
    "password": (
        "IT Help Guide. Every student receives one university account "
        "that is used for email, the learning portal and the library. "
        "Reset a forgotten password using the recovery link. The link is "
        "shown on the login page and is sent to the registered phone "
        "number or the alternate email address. Never share recovery "
        "codes with anyone, including friends, classmates or staff. "
        "Accounts locked after five failed attempts cannot use the "
        "recovery link and must be unlocked by the help desk. The help "
        "desk asks for the student card and a short description of the "
        "problem, and it does not ask for the current password at any "
        "time. After a successful reset the student should log out of "
        "all shared computers, choose a new password that has at least "
        "twelve characters, and avoid using the same password for "
        "personal accounts outside the university. Passwords expire "
        "every one hundred and eighty days, and a reminder is sent "
        "fourteen days before the expiry date."),
    "assignment": (
        "Submission Guide. Each course uses the same method for "
        "handing in written work so that files are never lost. Upload "
        "assignments through the learning portal. Submit the final file "
        "as a PDF and write the course code and student number in the "
        "file name. Students who cannot access the portal may email the "
        "file to the coordinator before the deadline and must keep the "
        "sent message as proof. The portal accepts files up to twenty "
        "megabytes, and larger files should be compressed or split into "
        "two parts that are clearly numbered. Each upload creates a "
        "receipt that shows the date and time, and students should save "
        "this receipt until the final marks are published. Only the "
        "latest upload before the deadline is marked, so a student may "
        "replace an earlier file as many times as needed. Source code, "
        "data files and images must be placed in one folder and "
        "submitted as a single archive together with the report. "
        "Plagiarism checks run automatically on every upload."),
}
PAIRS = {
    "late": ("Late submission is normally penalized by ten percent for "
             "each day after the deadline.",
             "Illness extensions require medical evidence before the "
             "deadline."),
    "password": ("Reset a forgotten password using the recovery link.",
                 "Accounts locked after five failed attempts cannot use "
                 "the recovery link and must be unlocked by the help desk."),
    "assignment": ("Upload assignments through the learning portal.",
                   "Students who cannot access the portal may email the "
                   "file to the coordinator before the deadline"),
}

if __name__ == "__main__":
    original_words = {d: len(t.split()) for d, t in DOCS.items()}
    for d, t in DOCS.items():
        longest = max(len(s.split()) for s in t.replace("?", ".").split("."))
        print(d, "words:", original_words[d],
              "| longest sentence (words):", longest)
        assert d in PAIRS and all(p in t for p in PAIRS[d])
    total_original = sum(original_words.values())
    print()

    for strategy, name in [("0", "window 40, overlap 0"),
                           ("10", "window 40, overlap 10"),
                           ("sentence", "sentence-aware 40")]:
        chunks = make_chunks(DOCS, strategy)
        sizes = [len(c["text"].split()) for c in chunks]
        duplicate = sum(sizes) - total_original
        survived = 0
        for d, (rule, exc) in PAIRS.items():
            if any(c["doc"] == d and rule in c["text"] and exc in c["text"]
                   for c in chunks):
                survived += 1
        print(f"{name}: chunks={len(chunks)}, "
              f"avg words={sum(sizes) / len(sizes):.1f}, "
              f"min={min(sizes)}, max={max(sizes)}, "
              f"duplicate words={duplicate}, "
              f"rule+exception in one chunk={survived}/{len(PAIRS)}")
        for c in chunks:
            assert c["text"] == DOCS[c["doc"]][c["start"]:c["end"]]
            print("   ", c["id"], c["start"], c["end"],
                  len(c["text"].split()), "words")
        print()
    print("Boundary tests")
    print("empty text:", windows(""), sentence_chunks(""))
    print("short document:", windows("short text"),
          sentence_chunks("short text"))
    try:
        windows("a b", 2, 2)
    except ValueError as error:
        print("invalid overlap -> ValueError:", error)
    long = " ".join(["word"] * 95) + "."
    spans = sentence_chunks(long)
    print("95-word sentence ->", len(spans), "chunks of",
          [len(long[a:b].split()) for a, b in spans], "words")

Writing task2_compare.py


In [5]:
!ls

lab6_search.py	sample_data  task2_chunks.py  task2_compare.py


In [6]:
%%writefile create_dataset.py
import json
from pathlib import Path
from task2_compare import DOCS
docs = dict(DOCS)
docs["contractor"] = (
    "Remote Work Policy. This policy explains where staff and contractors "
    "may work and how requests are approved. Employees may work remotely "
    "up to three days each week with the approval of their manager. "
    "Contractors can work remotely only if the signed contract permits it. "
    "Employee rules do not apply to contractors, so a manager cannot "
    "approve remote work for a contractor on a verbal basis. A contractor "
    "whose contract is silent on the matter must work from the office "
    "until the contract is amended in writing by the human resources "
    "office. Version one of this policy, which is now withdrawn, allowed "
    "contractors to work remotely for two days each week without any "
    "contract clause, and any older copy that says this must not be used. "
    "Requests for an amendment are sent to the human resources office "
    "together with the contract number, and the office replies within "
    "ten working days. Remote workers must stay reachable during office "
    "hours, use only approved company laptops, and connect through the "
    "secure network at all times.")
docs["printer"] = (
    "Printer Troubleshooting Guide. This guide covers the shared office "
    "printers on every floor and the most common faults reported to the "
    "help desk. For blank printer pages, check ink levels and run the "
    "cleaning utility. Print a test page afterwards to confirm that the "
    "fault is fixed, and note the result in the maintenance log. If the "
    "test page is still blank, remove the cartridge, check the protective "
    "tape and place it back firmly before trying again. For a blank "
    "scanner image, which is a different fault, check the document "
    "placement and select the correct scanning source instead of using "
    "these steps. Older laser models in the storage room use a drum unit, "
    "and for those models the guide says to replace the drum when pages "
    "are blank, but this advice does not apply to the current inkjet "
    "printers. Never open the printer while it is printing, and always "
    "switch the device off before removing a jammed sheet. Report any "
    "repeated fault to the help desk with the printer label and the time "
    "of the failure.")
evidence = {
    "late": "Illness extensions require medical evidence before the deadline.",
    "password": "Reset a forgotten password using the recovery link.",
    "assignment": "Upload assignments through the learning portal.",
    "contractor": "Contractors can work remotely only if the signed contract permits it.",
    "printer": "For blank printer pages, check ink levels and run the cleaning utility.",
}

questions = [
    ("What is required for illness extensions?", "late", "dev"),
    ("How do I reset a forgotten password?", "password", "dev"),
    ("Where do I upload assignments?", "assignment", "dev"),
    ("Can contractors work remotely?", "contractor", "dev"),
    ("What do I do for blank printer pages?", "printer", "test"),
    ("How can I recover my login access?", "password", "test"),
    ("How should I hand in my coursework?", "assignment", "test"),
    ("What proof is needed when I was ill?", "late", "test"),
    ("What are tomorrow's flight prices?", None, "test"),
    ("Who won the football match?", None, "test")]
queries = [dict(query=q, doc=doc,
                evidence=evidence[doc] if doc else None, split=split)
           for q, doc, split in questions]
for d, e in evidence.items():
    assert e in docs[d], d
    print(d, len(docs[d].split()), "words")
for name, data in [("corpus.json", docs), ("queries.json", queries)]:
    if Path(name).exists():
        raise FileExistsError("Refusing to overwrite " + name)
    Path(name).write_text(json.dumps(data, ensure_ascii=False, indent=2),
                          encoding="utf-8")
print("Created corpus.json and queries.json")

Writing create_dataset.py


In [7]:
!python create_dataset.py

late 173 words
password 160 words
assignment 165 words
contractor 171 words
printer 176 words
Created corpus.json and queries.json


In [8]:
%%writefile task3_retrieval.py
import csv, json, time
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from task2_chunks import make_chunks
def evaluate(scores, chunks, items):
    hits, reciprocal = [], []
    for row, q in zip(scores, items):
        if q["doc"] is None:
            continue
        top = np.argsort(-row, kind="stable")[:3]
        good = [rank for rank, i in enumerate(top, 1)
                if chunks[i]["doc"] == q["doc"]
                and q["evidence"] in chunks[i]["text"]]
        hits.append(bool(good))
        reciprocal.append(1 / good[0] if good else 0)
    return (float(np.mean(hits)), float(np.mean(reciprocal))) \
        if hits else (None, None)
def main():
    with open("corpus.json", encoding="utf-8") as f:
        docs = json.load(f)
    with open("queries.json", encoding="utf-8") as f:
        items = json.load(f)
    assert docs and items
    for q in items:
        assert q["split"] in {"dev", "test"}
        if q["doc"] is not None:
            assert q["evidence"] and q["evidence"] in docs[q["doc"]]
    t = time.perf_counter()
    model = SentenceTransformer(
        "sentence-transformers/all-MiniLM-L6-v2", device="cpu")
    print("Model load seconds", time.perf_counter() - t)
    query_texts = [q["query"] for q in items]
    logs = []
    for strategy in ["0", "10", "sentence"]:
        chunks = make_chunks(docs, strategy)
        texts = [c["text"] for c in chunks]
        lengths = [len(model.tokenizer(t, truncation=False)["input_ids"])
                   for t in texts + query_texts]
        print("Strategy", strategy, "| chunks:", len(chunks),
              "| max model tokens:", max(lengths),
              "| limit:", model.max_seq_length)
        if max(lengths) > model.max_seq_length:
            raise ValueError("Model limit exceeded; rechunk or shorten query")
        for method in ["tfidf", "embedding"]:
            t = time.perf_counter()
            if method == "tfidf":
                vectorizer = TfidfVectorizer()
                C = vectorizer.fit_transform(texts)
                Q = vectorizer.transform(query_texts)
            else:
                C = model.encode(texts, normalize_embeddings=True)
                Q = model.encode(query_texts, normalize_embeddings=True)
            encode_seconds = time.perf_counter() - t
            t = time.perf_counter()
            scores = ((Q @ C.T).toarray() if method == "tfidf"
                      else Q @ C.T)
            search_seconds = time.perf_counter() - t
            for split in ["dev", "test"]:
                indices = [i for i, q in enumerate(items)
                           if q["split"] == split]
                result = evaluate(scores[indices], chunks,
                                  [items[i] for i in indices])
                print(strategy, method, split, "Hit/MRR", result,
                      "chunks", len(chunks), "encode/search seconds",
                      encode_seconds, search_seconds)
            for row, q in zip(scores, items):
                for rank, i in enumerate(
                        np.argsort(-row, kind="stable")[:3], 1):
                    c = chunks[i]
                    record = dict(strategy=strategy, method=method,
                                  query=q["query"], split=q["split"],
                                  rank=rank, score=float(row[i]),
                                  chunk_id=c["id"], doc=c["doc"],
                                  start=c["start"], end=c["end"],
                                  text=c["text"],
                                  answerable=q["doc"] is not None)
                    logs.append(record)
                    print(strategy, method, repr(q["query"]), rank,
                          f"{record['score']:.3f}", c["id"],
                          c["start"], c["end"], repr(c["text"][:70]))
    with open("retrieval_log.csv", "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(logs[0]))
        writer.writeheader()
        writer.writerows(logs)
    print("Saved retrieval_log.csv with", len(logs), "rows")
if __name__ == "__main__":
    main()

Writing task3_retrieval.py


In [9]:
!python task3_retrieval.py

modules.json: 100% 349/349 [00:00<00:00, 1.44MB/s]
config_sentence_transformers.json: 100% 116/116 [00:00<00:00, 626kB/s]
README.md: 100% 10.5k/10.5k [00:00<00:00, 24.8MB/s]
sentence_bert_config.json: 100% 53.0/53.0 [00:00<00:00, 332kB/s]
config.json: 100% 612/612 [00:00<00:00, 3.49MB/s]

model.safetensors: downloading bytes:  60% 54.6M/90.9M [00:00<00:00, 105MB/s, 1.71MB/s  ] 
model.safetensors: downloading bytes: 100% 85.0M/85.0M [00:01<00:00, 75.6MB/s, 8.20MB/s  ]
model.safetensors: reconstructing file: 100% 90.9M/90.9M [00:01<00:00, 80.9MB/s, 8.87MB/s  ]
Loading weights: 100% 103/103 [00:00<00:00, 6653.32it/s]
tokenizer_config.json: 100% 350/350 [00:00<00:00, 2.00MB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 10.3MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 37.4MB/s]
special_tokens_map.json: 100% 112/112 [00:00<00:00, 567kB/s]
config.json: 100% 190/190 [00:00<00:00, 758kB/s]
Model load seconds 3.924421076000044
Strategy 0 | chunks: 24 | max model tokens: 50 | limit: 256
0 tfid

In [10]:
%%writefile document_chatbot.py
import json
from task2_chunks import make_chunks
from lab6_search import Retriever
if __name__ == "__main__":
    with open("corpus.json", encoding="utf-8") as f:
        docs = json.load(f)
    engine = Retriever(make_chunks(docs, "sentence"), "embedding")
    print("Evidence assistant. Type exit to stop.")
    while True:
        try:
            query = input("Question: ")
        except (EOFError, KeyboardInterrupt):
            break
        if query.strip().lower() == "exit":
            break
        hits = engine.search(query)
        if not hits:
            print("No candidate evidence; rephrase or contact a human.")
            continue
        print("Candidate evidence; relevance must be checked:")
        for number, (r, score) in enumerate(hits, 1):
            print(number, r["id"], f"{score:.3f}",
                  "characters", r["start"], r["end"])
        chosen = input("Choose 1-3 to inspect; Enter if none apply: ")
        if not chosen.isdigit() or not 1 <= int(chosen) <= len(hits):
            print("No answer selected. Rephrase or contact a human.")
            continue
        r, score = hits[int(chosen) - 1]
        print("Retrieved passage:", r["text"])
        print("Source:", r["doc"], "characters", r["start"], r["end"])

Writing document_chatbot.py


In [11]:
%run document_chatbot.py

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Evidence assistant. Type exit to stop.
Question: How can I recover my login access?
Candidate evidence; relevance must be checked:
1 password:4 0.524 characters 614 830
2 password:2 0.503 characters 365 476
3 password:0 0.490 characters 0 174
Choose 1-3 to inspect; Enter if none apply: What are tomorrow's flight prices?
No answer selected. Rephrase or contact a human.
Question: What are tomorrow's flight prices?
Candidate evidence; relevance must be checked:
1 password:5 0.074 characters 830 942
2 late:0 0.069 characters 0 244
3 late:4 0.066 characters 813 897
Choose 1-3 to inspect; Enter if none apply: 2
Retrieved passage: Assessment Policy. Assignments are due at 11:59 PM on the date stated in the course outline. Late submission is normally penalized by ten percent for each day after the deadline. Illness extensions require medical evidence before the deadline. 
Source: late characters 0 244


In [14]:
from google.colab import files
files.download("corpus.json")
files.download("queries.json")
files.download("retrieval_log.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>